# Módulo 1 — Simulación de Monte Carlo
## Rentabilidad del cultivo de pistacho — Jocolí, Mendoza (20 años)

Variables estocásticas: horas de frío · vecería · precio de mercado · falla de plantas  
Parámetros climáticos calibrados con datos históricos 1990–2024 (Módulo 0)

In [ ]:
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO_URL = "https://github.com/Emilialandgrebe/tesis-maestria.git"
    ROOT = Path("/content/tesis-maestria")
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt"), "-q"],
        check=True,
    )
else:
    ROOT = Path().resolve().parent  # notebooks/ -> raiz del proyecto

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Entorno : {'Google Colab' if EN_COLAB else 'local'}")
print(f"ROOT    : {ROOT}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

from src.monte_carlo import ParametrosMC, run_monte_carlo, CURVA_PRODUCCION, RENDIMIENTO_PLENA_KG_HA

np.random.seed(42)

params = ParametrosMC()
df = run_monte_carlo(params)

print(f"Simulaciones: {params.n_simulaciones:,}")
print(f"Años:         {params.n_años}")
print(f"Escenario:    {params.escenario}")
print(f"Filas totales: {len(df):,}")
df.head()

## Resumen estadístico — año 20 (plena producción)

In [ ]:
año20 = df[df["año"] == 20]

print("RENDIMIENTO kg/ha — año 20")
print(año20["rendimiento_kg_ha"].describe().round(1))
print()
print("INGRESO USD — año 20")
print(año20["ingreso_usd"].describe().round(0))
print()
print("VAN ACUMULADO USD — al año 20")
print(año20["van_neto_usd"].describe().round(0))
print()
p_van_neg = (año20["van_neto_usd"] < 0).mean()
print(f"Probabilidad de VAN negativo al año 20: {p_van_neg:.1%}")

## Evolución del rendimiento por año — percentiles 10/50/90

In [ ]:
pcts = df.groupby("año")["rendimiento_kg_ha"].quantile([0.10, 0.25, 0.50, 0.75, 0.90]).unstack()
años = pcts.index
curva_det = [CURVA_PRODUCCION[a] * RENDIMIENTO_PLENA_KG_HA for a in años]

fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(años, pcts[0.10], pcts[0.90], alpha=0.15, color="#2e86ab", label="P10–P90")
ax.fill_between(años, pcts[0.25], pcts[0.75], alpha=0.30, color="#2e86ab", label="P25–P75")
ax.plot(años, pcts[0.50], color="#2e86ab", linewidth=2.2, label="Mediana (P50)")
ax.plot(años, curva_det, color="#e07b39", linewidth=1.5, linestyle="--",
        label="Curva determinista (plan de negocios)")

ax.set_title("Rendimiento simulado — kg/ha por año del proyecto\n"
             f"Monte Carlo N={params.n_simulaciones:,} · escenario {params.escenario}",
             fontsize=13)
ax.set_xlabel("Año del proyecto")
ax.set_ylabel("kg/ha")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

## Evolución de ingresos brutos por año — USD

In [ ]:
pcts_ing = df.groupby("año")["ingreso_usd"].quantile([0.10, 0.25, 0.50, 0.75, 0.90]).unstack()
años = pcts_ing.index

fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(años, pcts_ing[0.10], pcts_ing[0.90], alpha=0.15, color="#2a9d8f", label="P10–P90")
ax.fill_between(años, pcts_ing[0.25], pcts_ing[0.75], alpha=0.30, color="#2a9d8f", label="P25–P75")
ax.plot(años, pcts_ing[0.50], color="#2a9d8f", linewidth=2.2, label="Mediana (P50)")

ax.set_title("Ingresos brutos simulados — USD por año del proyecto\n"
             f"Monte Carlo N={params.n_simulaciones:,} · escenario {params.escenario} · "
             f"{params.hectareas:.0f} ha",
             fontsize=13)
ax.set_xlabel("Año del proyecto")
ax.set_ylabel("USD")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

## Distribución del VAN acumulado al año 20

In [ ]:
van_final = df[df["año"] == 20]["van_neto_usd"] / 1e6

p10, p50, p90 = van_final.quantile([0.10, 0.50, 0.90])
prob_neg = (van_final < 0).mean()

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(van_final, bins=80, color="#2e86ab", alpha=0.75, edgecolor="none")
ax.axvline(0,   color="#c0392b", linewidth=2,   linestyle="-",  label="VAN = 0")
ax.axvline(p10, color="#e07b39", linewidth=1.5, linestyle="--", label=f"P10 = ${p10:.1f}M")
ax.axvline(p50, color="white",   linewidth=2,   linestyle="-",  label=f"P50 = ${p50:.1f}M")
ax.axvline(p90, color="#2a9d8f", linewidth=1.5, linestyle="--", label=f"P90 = ${p90:.1f}M")

ax.set_title(f"Distribución del VAN de ingresos al año 20\n"
             f"Escenario {params.escenario} · tasa descuento {params.tasa_descuento:.0%} · "
             f"P(VAN < 0) = {prob_neg:.1%}",
             fontsize=13)
ax.set_xlabel("VAN acumulado (millones USD)")
ax.set_ylabel("Frecuencia")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

## Comparación de los tres escenarios de precio

In [ ]:
colores = {"pesimista": "#e07b39", "base": "#2e86ab", "optimista": "#2a9d8f"}
resultados = {}

for escenario in ["pesimista", "base", "optimista"]:
    p = ParametrosMC(escenario=escenario)
    res = run_monte_carlo(p)
    resultados[escenario] = res[res["año"] == 20]["van_neto_usd"] / 1e6

fig, ax = plt.subplots(figsize=(11, 5))

for escenario, van in resultados.items():
    ax.hist(van, bins=60, alpha=0.55, color=colores[escenario],
            edgecolor="none", label=f"{escenario.capitalize()} · P50=${van.median():.1f}M")

ax.axvline(0, color="black", linewidth=1.5, linestyle="--", label="VAN = 0")
ax.set_title("VAN de ingresos al año 20 — comparación de escenarios de precio",
             fontsize=13)
ax.set_xlabel("VAN acumulado (millones USD)")
ax.set_ylabel("Frecuencia")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

print("\nP(VAN < 0) por escenario:")
for escenario, van in resultados.items():
    print(f"  {escenario:10s}: {(van < 0).mean():.1%}")